# Cell 1. 기본 경로 및 환경 설정

사용하는 내부 모듈:
src/utils/path_utils.py
- find_project_root

src/utils/file_utils.py
- load_jsonl

src/utils/eval_dataset_utils.py
- load_json
- save_json
- create_and_save_eval_sample

src/utils/progress_utils.py
- ProgressLogger
- progress_iter
- log_step

src/evaluation/rag_evaluator.py
- RAGEvaluator

처음 실행할 때:
FORCE_REBUILD_INDEX = True

다음부터 같은 청크와 같은 임베딩 모델을 재사용할 때:
FORCE_REBUILD_INDEX = False

단, 아래를 바꾸면 반드시 FAISS를 다시 만들어야 합니다.
section_chunks.jsonl 변경
embedding_model 변경
청크 정제/청킹 방식 변경

그 경우:
FORCE_REBUILD_INDEX = True

In [1]:
from pathlib import Path
import sys
import json
import pickle
import time
from typing import List, Dict, Any

import numpy as np
import pandas as pd

# 현재 노트북 위치:
# RFP-RAG-Extractor/notebooks/
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# src import를 위해 프로젝트 루트를 sys.path에 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.path_utils import find_project_root
from src.utils.file_utils import load_jsonl
from src.utils.eval_dataset_utils import (
    load_json,
    save_json,
    create_and_save_eval_sample,
)
from src.utils.progress_utils import ProgressLogger, progress_iter, log_step
from src.evaluation.evaluator import RAGEvaluator

# 프로젝트 이름 기준으로 root 검증
PROJECT_ROOT = find_project_root("RFP-RAG-Extractor")

DATA_DIR = PROJECT_ROOT / "data"

# 01번 노트북에서 생성한 청크 파일
SECTION_CHUNK_PATH = DATA_DIR / "chunks" / "section" / "section_chunks.jsonl"

# 평가 데이터 경로
EVAL_DIR = DATA_DIR / "processed" / "eval"
EVAL_DATASET_PATH = EVAL_DIR / "eval_dataset.json"
EVAL_SAMPLE_PATH = EVAL_DIR / "eval_dataset_sample_20.json"

# 벡터 DB 저장 경로
VECTOR_DB_DIR = DATA_DIR / "vector_db"
BASELINE_VECTOR_DIR = VECTOR_DB_DIR / "baseline_section_kure_faiss"
BASELINE_VECTOR_DIR.mkdir(parents=True, exist_ok=True)

# RAG 실행 결과 저장 경로
RAG_OUTPUT_PATH = EVAL_DIR / "rag_outputs_baseline_section_sample_20.json"
RAG_OUTPUT_SCORED_PATH = EVAL_DIR / "rag_outputs_baseline_section_sample_20_scored.json"

# 평가 리포트 저장 경로
REPORT_DIR = PROJECT_ROOT / "reports" / "evaluation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = REPORT_DIR / "baseline_section_sample20_metrics.json"
METRICS_BY_QTYPE_PATH = REPORT_DIR / "baseline_section_sample20_by_question_type.json"
METRICS_BY_SOURCE_TYPE_PATH = REPORT_DIR / "baseline_section_sample20_by_source_type.json"
METRICS_BY_ANSWER_FORMAT_PATH = REPORT_DIR / "baseline_section_sample20_by_answer_format.json"
METRICS_BY_FILE_TYPE_PATH = REPORT_DIR / "baseline_section_sample20_by_file_type.json"

RETRIEVAL_FAILURE_PATH = REPORT_DIR / "baseline_section_sample20_retrieval_failures.csv"
KEYWORD_FAILURE_PATH = REPORT_DIR / "baseline_section_sample20_keyword_failures.csv"
SUMMARY_CSV_PATH = REPORT_DIR / "baseline_section_sample20_summary.csv"

print("현재 작업 위치:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SECTION_CHUNK_PATH:", SECTION_CHUNK_PATH)
print("EVAL_DATASET_PATH:", EVAL_DATASET_PATH)
print("EVAL_SAMPLE_PATH:", EVAL_SAMPLE_PATH)
print("BASELINE_VECTOR_DIR:", BASELINE_VECTOR_DIR)

현재 작업 위치: /home/user1/RFP-RAG-Extractor/notebooks
PROJECT_ROOT: /home/user1/RFP-RAG-Extractor
SECTION_CHUNK_PATH: /home/user1/RFP-RAG-Extractor/data/chunks/section/section_chunks.jsonl
EVAL_DATASET_PATH: /home/user1/RFP-RAG-Extractor/data/processed/eval/eval_dataset.json
EVAL_SAMPLE_PATH: /home/user1/RFP-RAG-Extractor/data/processed/eval/eval_dataset_sample_20.json
BASELINE_VECTOR_DIR: /home/user1/RFP-RAG-Extractor/data/vector_db/baseline_section_kure_faiss


# Cell 2. 베이스라인 설정

In [2]:
BASELINE_CONFIG = {
    "llm_model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "embedding_model_name": "nlpai-lab/KURE-v1",
    "vector_db": "FAISS",
    "chunking_strategy": "section",
    "top_k": 5,
    "sample_size": 20,
    "random_seed": 42,
    "max_new_tokens": 512,
    "temperature": 0.0,
    "do_sample": False,
}

# section_chunks.jsonl을 새로 만들었으면 True
# 이미 같은 청크로 FAISS 인덱스를 만들어뒀으면 False
FORCE_REBUILD_INDEX = True

BASELINE_CONFIG

{'llm_model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
 'embedding_model_name': 'nlpai-lab/KURE-v1',
 'vector_db': 'FAISS',
 'chunking_strategy': 'section',
 'top_k': 5,
 'sample_size': 20,
 'random_seed': 42,
 'max_new_tokens': 512,
 'temperature': 0.0,
 'do_sample': False}

# Cell 3. 평가 데이터셋 로드 및 샘플 n개 생성

In [3]:
if not EVAL_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"평가 데이터셋이 없습니다: {EVAL_DATASET_PATH}"
    )

eval_dataset = load_json(EVAL_DATASET_PATH)

print("전체 평가 문항 수:", len(eval_dataset))
print(pd.Series([x.get("question_type") for x in eval_dataset]).value_counts())

전체 평가 문항 수: 243
llm_1          82
llm_2          82
fact_budget    79
Name: count, dtype: int64


In [4]:
if EVAL_SAMPLE_PATH.exists():
    print("기존 샘플 평가셋 로드:", EVAL_SAMPLE_PATH)
    sample_eval_dataset = load_json(EVAL_SAMPLE_PATH)

else:
    print("샘플 평가셋 새로 생성:", EVAL_SAMPLE_PATH)
    sample_eval_dataset = create_and_save_eval_sample(
        input_path=EVAL_DATASET_PATH,
        output_path=EVAL_SAMPLE_PATH,
        sample_size=BASELINE_CONFIG["sample_size"],
        random_seed=BASELINE_CONFIG["random_seed"],
    )

print("샘플 문항 수:", len(sample_eval_dataset))
print(pd.Series([x.get("question_type") for x in sample_eval_dataset]).value_counts())

sample_eval_dataset[:2]

기존 샘플 평가셋 로드: /home/user1/RFP-RAG-Extractor/data/processed/eval/eval_dataset_sample_20.json
샘플 문항 수: 20
llm_1          8
fact_budget    6
llm_2          6
Name: count, dtype: int64


[{'qid': '20240330003_llm_1',
  'question': '2024년 벤처확인종합관리시스템 기능 고도화 사업에서 고도화하거나 신규로 구축하는 주요 기능은 무엇인가요?',
  'reference': '복수의결권주식의 발행 보고 업무처리 시스템 구축, 벤처기업법에 따라 부여된 스톡옵션(주식매수선택권)의 부여·취소·철회 신고 및 업무시스템 구축, 성과조건부주식교부계약(RS)의 신고 및 업무처리 시스템 구축 등 세 가지 기능의 구축 및 고도화이다.',
  'required_keyword_groups': [['복수의결권주식', '복수 의결권'],
   ['스톡옵션', '주식매수선택권', '스톡 옵션'],
   ['성과조건부주식교부', '성과조건부주식교부계약', 'RS']],
  'doc_id': '20240330003',
  'question_type': 'llm_1',
  'source_type': 'document',
  'answer_format': 'list',
  'source_row_id': 86,
  'project_name': '2024년 벤처확인종합관리시스템 기능 고도화 용역사업 입찰공고',
  'organization': '(사)벤처기업협회',
  'file_name': '(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp',
  'file_type': 'hwp'},
 {'qid': '20240404154_fact_budget',
  'question': "서울특별시에서 발주한 '2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역' 사업의 배정 예산은 얼마인가요?",
  'reference': '493,763,000원 입니다.',
  'required_keyword_groups': [['493,763,000원', '493763000원', '4억9천3백7십6만원']],
  'doc_id': '20240404154',
  'question_type': 'fact_budget',
  'sourc

# Cell 4. section_chunks.jsonl 로드

In [5]:
if not SECTION_CHUNK_PATH.exists():
    raise FileNotFoundError(
        f"section_chunks.jsonl 파일이 없습니다: {SECTION_CHUNK_PATH}\n"
        "먼저 01_extract_clean_chunk.ipynb를 실행해서 청크를 생성해야 합니다."
    )

chunks = load_jsonl(SECTION_CHUNK_PATH)

print("로드된 청크 수:", len(chunks))
print("첫 번째 청크 keys:", chunks[0].keys())
chunks[0]

로드된 청크 수: 14415
첫 번째 청크 keys: dict_keys(['chunk_id', 'doc_id', 'file_name', 'file_type', 'project_name', 'organization', 'chunking_strategy', 'section_title', 'section_path', 'text'])


{'chunk_id': '20241001798_section_0024_00',
 'doc_id': '20241001798',
 'file_name': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
 'file_type': 'hwp',
 'project_name': '한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화',
 'organization': '한영대학',
 'chunking_strategy': 'section',
 'section_title': '□ 입찰방법 : 제한경쟁입찰(협상에 의한 계약 체결)',
 'section_path': ['□ 입찰방법 : 제한경쟁입찰(협상에 의한 계약 체결)'],
 'text': '□ 입찰방법 : 제한경쟁입찰(협상에 의한 계약 체결)\n\n氠瑢\n\n2\n\n추진배경 및 필요성\n\n학사제도·제도개편과 연계하湯湷여 전공교과목 선택폭을 넓히고, 트랙제 교육과정 참여자에게 다양한 진로선택의 기회를 제공 및 취업문 확대\n\n트랙제 교육과정의 도입 및 운영으로 산업현장의 경쟁력 강화\n\n산업체 수요 맞춤 교육과정 운영 및 활성화로 교육과정 내실화\n\n기업수요 연계 확대로 산업체 및 지역사회 현장실무형 인재 양성\n\n氠瑢\n\n3\n\n기대효과\n\n◦ 트랙기반 교육과정의 운영 및 관리 체계를 효과적으로 지원\n\n◦ 교수자·학습자 중심의 교육환경 조성을 통한 대학 교육의 가치 구현\n\n◦ 학사운영 시스템을 통해 대학 체제 개편에 대한 대응체계 확립\n\n氠瑢\n\nII\n\n구축 방안\n\n氠瑢\n\n1\n\n구축목표\n\n◦ 트랙제도 교과과정 개편 및 표준운영관리를 위한 시스템 구현\n\n(교과과정 개발에서 성적 이수까지)\n\n◦ 트랙제도 기반 교육과정과 현행 종합정보시스템 연계 가능하도록 고도화\n\n(교육과정 / 수강신청 / 성적 / 학적관리 외)\n\n◦ 확장성과 유연성을 고려한 환경 조성 및 유지보수가 용이 하도록 해야 함.\n\n◦ 다양한 사용자(교수

# Cell 5. 청크 필드 표준화

In [6]:
def get_chunk_text(chunk: Dict[str, Any]) -> str:
    for key in ["text", "page_content", "content", "chunk_text"]:
        value = chunk.get(key)
        if value:
            return str(value)
    return ""


def get_chunk_doc_id(chunk: Dict[str, Any]) -> str:
    if chunk.get("doc_id") is not None:
        return str(chunk["doc_id"])

    metadata = chunk.get("metadata", {}) or {}
    if metadata.get("doc_id") is not None:
        return str(metadata["doc_id"])

    return ""


def get_chunk_id(chunk: Dict[str, Any], idx: int) -> str:
    if chunk.get("chunk_id") is not None:
        return str(chunk["chunk_id"])

    metadata = chunk.get("metadata", {}) or {}
    if metadata.get("chunk_id") is not None:
        return str(metadata["chunk_id"])

    return f"chunk_{idx:06d}"


standard_chunks = []

for idx, chunk in enumerate(chunks):
    text = get_chunk_text(chunk)
    doc_id = get_chunk_doc_id(chunk)
    chunk_id = get_chunk_id(chunk, idx)

    if not text.strip():
        continue

    if not doc_id.strip():
        continue

    standard_chunks.append({
        **chunk,
        "chunk_id": chunk_id,
        "doc_id": doc_id,
        "text": text,
    })

print("표준화 전 청크 수:", len(chunks))
print("표준화 후 청크 수:", len(standard_chunks))

pd.DataFrame([
    {
        "chunk_id": x["chunk_id"],
        "doc_id": x["doc_id"],
        "file_type": x.get("file_type"),
        "text_len": len(x["text"]),
    }
    for x in standard_chunks[:5]
])

표준화 전 청크 수: 14415
표준화 후 청크 수: 14415


,chunk_id,doc_id,file_type,text_len
0,20241001798_section_0024_00,20241001798,hwp,770
1,20241001798_section_0036_00,20241001798,hwp,126
2,20241001798_section_0037_00,20241001798,hwp,1114
3,20241001798_section_0044_00,20241001798,hwp,780
4,20241001798_section_0057_00,20241001798,hwp,111


# Cell 6. 청크 통계 확인

In [7]:
chunk_stat_df = pd.DataFrame([
    {
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "file_type": chunk.get("file_type"),
        "chunking_strategy": chunk.get("chunking_strategy"),
        "text_len": len(chunk.get("text", "")),
    }
    for chunk in standard_chunks
])

print("청크 수:", len(chunk_stat_df))
display(chunk_stat_df["text_len"].describe())
display(chunk_stat_df.groupby("doc_id")["chunk_id"].count().describe())
display(chunk_stat_df["file_type"].value_counts(dropna=False))

청크 수: 14415


count    14415.000000
mean       380.335831
std        459.127526
min        100.000000
25%        136.000000
50%        214.000000
75%        405.000000
max       3000.000000
Name: text_len, dtype: float64

count    100.00000
mean     144.15000
std       63.24162
min       51.00000
25%      105.00000
50%      130.00000
75%      165.25000
max      481.00000
Name: chunk_id, dtype: float64

file_type
hwp    13267
pdf     1148
Name: count, dtype: int64

# Cell 6. 임베딩 모델 로드

In [8]:
from sentence_transformers import SentenceTransformer

embedding_model_name = BASELINE_CONFIG["embedding_model_name"]

with log_step(f"Embedding model load: {embedding_model_name}"):
    embedding_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded:", embedding_model_name)

[Embedding model load: nlpai-lab/KURE-v1] start


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[Embedding model load: nlpai-lab/KURE-v1] done | elapsed=4s
Embedding model loaded: nlpai-lab/KURE-v1


# Cell 8. FAISS 인덱스 생성 함수

In [9]:
import faiss


def build_faiss_index(
    chunks: List[Dict[str, Any]],
    embedding_model: SentenceTransformer,
    batch_size: int = 32,
    log_every: int = 10,
):
    """
    청크 텍스트를 임베딩하고 FAISS IndexFlatIP 인덱스를 생성합니다.

    JupyterHub/GCP 환경에서 tqdm 렌더링 문제가 있을 수 있으므로
    ProgressLogger 기반으로 진행 상황을 출력합니다.
    """
    texts = [chunk["text"] for chunk in chunks]

    total = len(texts)
    num_batches = (total + batch_size - 1) // batch_size

    logger = ProgressLogger(
        total=num_batches,
        desc="Embedding batches",
        log_every=log_every,
        min_interval_sec=5.0,
    )

    all_embeddings = []

    logger.start()

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch_texts = texts[start:end]

        batch_embeddings = embedding_model.encode(
            batch_texts,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        all_embeddings.append(batch_embeddings)

        logger.update(
            1,
            message=f"chunks={end}/{total}"
        )

    logger.done(message="embedding complete")

    embeddings = np.vstack(all_embeddings).astype("float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    print("FAISS index 생성 완료")
    print("embedding shape:", embeddings.shape)
    print("FAISS ntotal:", index.ntotal)

    return index, embeddings

# Cell 9. FAISS 인덱스 생성/로드

In [10]:
FAISS_INDEX_PATH = BASELINE_VECTOR_DIR / "index.faiss"
CHUNK_META_PATH = BASELINE_VECTOR_DIR / "chunks.pkl"
EMBEDDING_CONFIG_PATH = BASELINE_VECTOR_DIR / "config.json"

if (
    not FORCE_REBUILD_INDEX
    and FAISS_INDEX_PATH.exists()
    and CHUNK_META_PATH.exists()
):
    with log_step("FAISS index load"):
        index = faiss.read_index(str(FAISS_INDEX_PATH))

        with open(CHUNK_META_PATH, "rb") as f:
            indexed_chunks = pickle.load(f)

else:
    print("FAISS 인덱스 새로 생성")

    index, embeddings = build_faiss_index(
        chunks=standard_chunks,
        embedding_model=embedding_model,
        batch_size=32,
        log_every=10,
    )

    indexed_chunks = standard_chunks

    with log_step("FAISS index save"):
        faiss.write_index(index, str(FAISS_INDEX_PATH))

        with open(CHUNK_META_PATH, "wb") as f:
            pickle.dump(indexed_chunks, f)

        save_json(BASELINE_CONFIG, EMBEDDING_CONFIG_PATH)

print("인덱스 청크 수:", len(indexed_chunks))
print("FAISS ntotal:", index.ntotal)

FAISS 인덱스 새로 생성
[Embedding batches] start | total=451
[Embedding batches] 1/451 (0.2%) | elapsed=4s | eta=34m 18s | chunks=32/14415
[Embedding batches] 4/451 (0.9%) | elapsed=10s | eta=19m 47s | chunks=128/14415
[Embedding batches] 6/451 (1.3%) | elapsed=16s | eta=20m 24s | chunks=192/14415
[Embedding batches] 8/451 (1.8%) | elapsed=22s | eta=20m 42s | chunks=256/14415
[Embedding batches] 10/451 (2.2%) | elapsed=24s | eta=17m 38s | chunks=320/14415
[Embedding batches] 12/451 (2.7%) | elapsed=29s | eta=18m 4s | chunks=384/14415
[Embedding batches] 14/451 (3.1%) | elapsed=34s | eta=18m 7s | chunks=448/14415
[Embedding batches] 16/451 (3.5%) | elapsed=42s | eta=19m 28s | chunks=512/14415
[Embedding batches] 19/451 (4.2%) | elapsed=50s | eta=19m 4s | chunks=608/14415
[Embedding batches] 20/451 (4.4%) | elapsed=51s | eta=18m 27s | chunks=640/14415
[Embedding batches] 21/451 (4.7%) | elapsed=56s | eta=19m 16s | chunks=672/14415
[Embedding batches] 25/451 (5.5%) | elapsed=1m 4s | eta=18m 11s 

# Cell 10. 검색 함수 구현

In [11]:
def retrieve(
    query: str,
    top_k: int = 5,
) -> List[Dict[str, Any]]:
    """
    질문을 임베딩한 뒤 FAISS에서 top_k 청크를 검색합니다.

    반환 결과의 doc_id는 RAGEvaluator의 retrieval 평가에 사용됩니다.
    """
    query_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    scores, indices = index.search(query_emb, top_k)

    results = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        if idx < 0:
            continue

        chunk = indexed_chunks[int(idx)]

        results.append({
            "rank": rank,
            "score": float(score),
            "chunk_id": chunk.get("chunk_id"),
            "doc_id": chunk.get("doc_id"),
            "text": chunk.get("text", ""),
            "metadata": {
                k: v
                for k, v in chunk.items()
                if k not in {"text"}
            },
        })

    return results

In [12]:
# Cell 11. 검색 단일 테스트

In [13]:
test_question = sample_eval_dataset[0]["question"]
retrieved = retrieve(test_question, top_k=BASELINE_CONFIG["top_k"])

print("질문:", test_question)
print("정답 doc_id:", sample_eval_dataset[0]["doc_id"])
print()

for item in retrieved:
    print(
        item["rank"],
        item["score"],
        item["doc_id"],
        item["chunk_id"],
        item["text"][:100].replace("\n", " ")
    )

질문: 2024년 벤처확인종합관리시스템 기능 고도화 사업에서 고도화하거나 신규로 구축하는 주요 기능은 무엇인가요?
정답 doc_id: 20240330003

1 0.6914679408073425 20240330003 20240330003_section_0550_00 □ 별지서식  <별지서식 1호> 2024년 벤처확인종합관리시스템 기능 고도화 용역사업 제안서 표지  <별지서식 2호> 제안요청서 수용 여부 참조표  <별지서식 3호> 일반현황 및 
2 0.6372312307357788 20240330003 20240330003_section_0018_00 □ 추진체계  氠瑢  중소벤처기업부  벤처기업확인기관  (시스템 고도화 발주)  자문위원  (법률, 검수·검사 등)  사업 수행업체  벤처확인종합관리시스템 기능 고도화 수행  (복
3 0.6277969479560852 20240330003 20240330003_section_0019_00 □ 추진역할  氠瑢  구 분  역 할  중소벤처기업부  ᆞ사업총괄 운영ᆞ관리 감독  벤처기업확인기관  汤捯ᆞ벤처확인제도 운영기관  ᆞ사업계획 수립, 제안요청서 작성  ᆞ사업관리 및
4 0.6157175302505493 20240330003 20240330003_section_0107_00 □ 요구사항 상세  氠瑢  가  기능 요구사항(System Function Requirement)  氠瑢  요구사항 고유번호  SFR-001  요구사항 분류  기능 요구사항  요구
5 0.6129816770553589 20240330003 20240330003_section_0551_00 □ 붙임자료  [붙임1] 청렴서약서  [붙임2] 근로자 권리보호 이행 서약서  [붙임3] 보안서약서  [붙임4] 정보화 용역사업 보안특약  [붙임5] 소프트웨어 개발사업의 적정 사


# Cell 12. Qwen LLM 로드

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_model_name = BASELINE_CONFIG["llm_model_name"]

print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

with log_step(f"Tokenizer load: {llm_model_name}"):
    tokenizer = AutoTokenizer.from_pretrained(
        llm_model_name,
        trust_remote_code=True,
    )

with log_step(f"LLM model load: {llm_model_name}"):
    model = AutoModelForCausalLM.from_pretrained(
        llm_model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )

model.eval()

print("LLM loaded:", llm_model_name)

CUDA 사용 가능: True
GPU: NVIDIA L4
[Tokenizer load: Qwen/Qwen2.5-1.5B-Instruct] start


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[Tokenizer load: Qwen/Qwen2.5-1.5B-Instruct] done | elapsed=0s
[LLM model load: Qwen/Qwen2.5-1.5B-Instruct] start


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[LLM model load: Qwen/Qwen2.5-1.5B-Instruct] done | elapsed=1s
LLM loaded: Qwen/Qwen2.5-1.5B-Instruct


# Cell 13. RFP RAG 프롬프트 구성

In [15]:
def format_context(retrieved_chunks: List[Dict[str, Any]]) -> str:
    """
    검색된 청크들을 LLM 프롬프트에 넣을 context 문자열로 변환합니다.
    """
    context_blocks = []

    for item in retrieved_chunks:
        metadata = item.get("metadata", {})

        doc_id = item.get("doc_id", "")
        chunk_id = item.get("chunk_id", "")
        score = item.get("score", "")
        text = item.get("text", "")

        section_title = metadata.get("section_title", "")
        section_path = metadata.get("section_path", "")

        block = f"""
[문서 {item['rank']}]
doc_id: {doc_id}
chunk_id: {chunk_id}
score: {score}
section_title: {section_title}
section_path: {section_path}

{text}
""".strip()

        context_blocks.append(block)

    return "\n\n---\n\n".join(context_blocks)


def build_messages(
    question: str,
    retrieved_chunks: List[Dict[str, Any]]
) -> List[Dict[str, str]]:
    """
    Qwen Instruct 모델용 chat messages를 구성합니다.
    """
    context = format_context(retrieved_chunks)

    system_prompt = """
당신은 기업 및 정부 제안요청서(RFP)를 분석하는 전문 RAG Assistant입니다.
반드시 제공된 문서 내용에 근거해서만 답변하세요.

규칙:
1. 문서에 근거가 있는 내용만 답변하세요.
2. 문서에서 확인할 수 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다."라고 답변하세요.
3. 질문이 예산, 기간, 계약방법, 담당자, 연락처처럼 정확한 값을 요구하면 값 중심으로 간결하게 답변하세요.
4. 질문이 사업범위, 추진목표, 기대효과처럼 목록형 답변을 요구하면 핵심 항목을 bullet로 정리하세요.
5. 가능하면 답변 마지막에 근거가 된 doc_id를 적으세요.
""".strip()

    user_prompt = f"""
아래는 검색된 RFP 문서 일부입니다.

[검색 문서]
{context}

[질문]
{question}

[답변]
""".strip()

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

# Cell 14. LLM 답변 생성 함수

In [16]:
def generate_answer(
    question: str,
    retrieved_chunks: List[Dict[str, Any]],
    max_new_tokens: int = 512,
) -> Dict[str, Any]:
    """
    검색된 청크와 질문을 바탕으로 LLM 답변을 생성합니다.

    반환:
    - response
    - input_tokens
    - output_tokens
    - total_tokens
    - generation_latency_sec
    - prompt_text
    """
    messages = build_messages(question, retrieved_chunks)

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    input_tokens = int(inputs["input_ids"].shape[-1])

    start_time = time.perf_counter()

    generation_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "do_sample": BASELINE_CONFIG["do_sample"],
        "pad_token_id": tokenizer.eos_token_id,
    }

    if BASELINE_CONFIG["do_sample"]:
        generation_kwargs["temperature"] = BASELINE_CONFIG["temperature"]

    with torch.no_grad():
        outputs = model.generate(**generation_kwargs)

    generation_latency_sec = time.perf_counter() - start_time

    output_tokens = int(outputs.shape[-1] - inputs["input_ids"].shape[-1])

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    response_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "response": response_text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "generation_latency_sec": generation_latency_sec,
        "prompt_text": prompt_text,
    }

# Cell 15. 단일 RAG 실행 함수

In [17]:
def run_single_rag(
    eval_item: Dict[str, Any],
    top_k: int = 5,
) -> Dict[str, Any]:
    """
    평가 문항 1개에 대해:
    1. 검색
    2. LLM 답변 생성
    3. 평가에 필요한 결과 row 생성
    """
    question = eval_item["question"]

    start_total = time.perf_counter()

    start_retrieval = time.perf_counter()
    retrieved_chunks = retrieve(question, top_k=top_k)
    retrieval_latency_sec = time.perf_counter() - start_retrieval

    generation_result = generate_answer(
        question=question,
        retrieved_chunks=retrieved_chunks,
        max_new_tokens=BASELINE_CONFIG["max_new_tokens"],
    )

    total_latency_sec = time.perf_counter() - start_total

    retrieved_ids = [
        item.get("doc_id")
        for item in retrieved_chunks
        if item.get("doc_id") is not None
    ]

    retrieved_contexts = [
        item.get("text", "")
        for item in retrieved_chunks
    ]

    result = {
        **eval_item,
        "retrieved_ids": retrieved_ids,
        "retrieved_chunks": [
            {
                "rank": item["rank"],
                "score": item["score"],
                "doc_id": item["doc_id"],
                "chunk_id": item["chunk_id"],
                "text": item["text"][:1500],
            }
            for item in retrieved_chunks
        ],
        "retrieved_contexts": retrieved_contexts,
        "response": generation_result["response"],
        "retrieval_latency_sec": retrieval_latency_sec,
        "generation_latency_sec": generation_result["generation_latency_sec"],
        "total_latency_sec": total_latency_sec,
        "input_tokens": generation_result["input_tokens"],
        "output_tokens": generation_result["output_tokens"],
        "total_tokens": generation_result["total_tokens"],

        # 현재는 로컬 모델이므로 API 비용은 0으로 둡니다.
        # API 모델 실험 시 RAGEvaluator.attach_costs()로 비용 계산 가능.
        "estimated_cost": 0.0,
    }

    return result

In [18]:
# Cell 16. 단일 질문 테스트

In [19]:
test_item = sample_eval_dataset[0]

test_result = run_single_rag(
    test_item,
    top_k=BASELINE_CONFIG["top_k"],
)

print("Q:", test_result["question"])
print()
print("A:", test_result["response"])
print()
print("Reference:", test_result["reference"])
print()
print("정답 doc_id:", test_result["doc_id"])
print("retrieved_ids:", test_result["retrieved_ids"])
print("latency:", test_result["total_latency_sec"])

Q: 2024년 벤처확인종합관리시스템 기능 고도화 사업에서 고도화하거나 신규로 구축하는 주요 기능은 무엇인가요?

A: 2024년 벤처확인종합관리시스템 기능 고도화 사업에서 고도화하거나 신규로 구축하는 주요 기능은 다음과 같습니다:

1. **사용자 인증 기능**: 복수의결권주식 신청자/창업주/대리인 확인절차 기능 기획 및 구현, 스톡옵션 신청자/부여(대상자) 확인절차 기능 기획 및 구현, 주민등록번호 미수집에 따른 대체 인증 방안 제시, 벤처확인종합관리시스템 회원통합을 위한 인증방안 제시 및 구현, 복수의결권주식, 스톡옵션 업무 담당자 인증.

2. **회원가입 기능**: 중소벤처24 스톡옵션 신청자 회원통합 방안/기획 및 기능 구현, 중소벤처24 Easy Pass 연계, 벤처확인종합관리시스템 이관사용자와의 회원통합, 중소벤처24 스톡옵션 신청자 회원통합 이력 조회 기능 구현.

3. **중소기업확인서 기능**: 복수의결권주식 발행보고 및 스톡옵션 부여신고 前 중소기업여부 확인, 중소기업확인서 발급정보 연동, 중소기업확인서 발급정보 연동 불가시 파일업로드, 복수의결권주식 발행보고 및 스톡옵션 부여신고 신청단계 중소기업 확인 기능 구현.

4. **비상장 여부 조회 기능**: 법인등록번호 기준 비상장 조회기능 구현.

이들 기능들은 벤처확인종합관리시스템의 효율성을 높이고, 사용자 인증, 회원 관리, 중소기업 확인 등의 중요한 기능을 강화함으로써 사업의 성공을 촉진합니다.

Reference: 복수의결권주식의 발행 보고 업무처리 시스템 구축, 벤처기업법에 따라 부여된 스톡옵션(주식매수선택권)의 부여·취소·철회 신고 및 업무시스템 구축, 성과조건부주식교부계약(RS)의 신고 및 업무처리 시스템 구축 등 세 가지 기능의 구축 및 고도화이다.

정답 doc_id: 20240330003
retrieved_ids: ['20240330003', '20240330003', '20240330003', '20240330003', '20240330003']
latency: 15.6

# Cell 17. 샘플 20개 전체 RAG 실행

In [20]:
rag_outputs = []

for item in progress_iter(
    sample_eval_dataset,
    total=len(sample_eval_dataset),
    desc="Running baseline RAG",
    log_every=1,
    min_interval_sec=0.0,
):
    try:
        result = run_single_rag(
            item,
            top_k=BASELINE_CONFIG["top_k"],
        )
        rag_outputs.append(result)

    except Exception as e:
        error_result = {
            **item,
            "retrieved_ids": [],
            "retrieved_chunks": [],
            "retrieved_contexts": [],
            "response": "",
            "error": repr(e),
            "retrieval_latency_sec": 0.0,
            "generation_latency_sec": 0.0,
            "total_latency_sec": 0.0,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "estimated_cost": 0.0,
        }
        rag_outputs.append(error_result)

save_json(rag_outputs, RAG_OUTPUT_PATH)

print("RAG 실행 결과 저장:", RAG_OUTPUT_PATH)
print("결과 수:", len(rag_outputs))

[Running baseline RAG] start | total=20
[Running baseline RAG] 1/20 (5.0%) | elapsed=15s | eta=4m 52s
[Running baseline RAG] 2/20 (10.0%) | elapsed=16s | eta=2m 27s
[Running baseline RAG] 3/20 (15.0%) | elapsed=17s | eta=1m 41s
[Running baseline RAG] 4/20 (20.0%) | elapsed=21s | eta=1m 27s
[Running baseline RAG] 5/20 (25.0%) | elapsed=23s | eta=1m 10s
[Running baseline RAG] 6/20 (30.0%) | elapsed=31s | eta=1m 14s
[Running baseline RAG] 7/20 (35.0%) | elapsed=32s | eta=1m 0s
[Running baseline RAG] 8/20 (40.0%) | elapsed=37s | eta=56s
[Running baseline RAG] 9/20 (45.0%) | elapsed=43s | eta=52s
[Running baseline RAG] 10/20 (50.0%) | elapsed=44s | eta=44s
[Running baseline RAG] 11/20 (55.0%) | elapsed=44s | eta=36s
[Running baseline RAG] 12/20 (60.0%) | elapsed=46s | eta=30s
[Running baseline RAG] 13/20 (65.0%) | elapsed=46s | eta=25s
[Running baseline RAG] 14/20 (70.0%) | elapsed=55s | eta=23s
[Running baseline RAG] 15/20 (75.0%) | elapsed=1m 1s | eta=20s
[Running baseline RAG] 16/20 (80.

In [21]:
# Cell 18. RAG 실행 결과 확인

In [22]:
rag_df = pd.DataFrame([
    {
        "qid": row.get("qid"),
        "doc_id": row.get("doc_id"),
        "question_type": row.get("question_type"),
        "response_len": len(row.get("response", "")),
        "retrieved_ids": row.get("retrieved_ids"),
        "total_latency_sec": row.get("total_latency_sec"),
        "input_tokens": row.get("input_tokens"),
        "output_tokens": row.get("output_tokens"),
        "error": row.get("error", ""),
    }
    for row in rag_outputs
])

display(rag_df)
print("에러 개수:", rag_df["error"].astype(bool).sum())

,qid,doc_id,question_type,response_len,retrieved_ids,total_latency_sec,input_tokens,output_tokens,error
0,20240330003_llm_1,20240330003,llm_1,654,"[20240330003, 20240330003, 20240330003, 202403...",15.377135,2235,465,
1,20240404154_fact_budget,20240404154,fact_budget,31,"[20240404154, 20240404154, 20240404154, 202404...",1.051589,2701,26,
2,20240408682_llm_1,20240408682,llm_1,54,"[20240408682, 20240408682, DOC_0026, 202404086...",1.444478,2221,39,
3,20240430896_llm_2,20240430896,llm_2,147,"[20240531160, DOC_0026, 20240903676, R25BK0056...",3.956837,2462,113,
4,20240430918_fact_budget,20240430918,fact_budget,86,"[20240430918, 20240430918, 20240430918, 202404...",1.525542,3451,38,
5,20240430918_llm_2,20240430918,llm_2,378,"[R25BK00604826, 20240414353, 20240419826, 2024...",8.499642,2769,248,
6,20240523741_llm_1,20240523741,llm_1,19,"[20240419826, 20240611568, 20241218257, 202410...",0.577121,1134,15,
7,20240531285_llm_1,20240531285,llm_1,232,"[20240531285, 20240531013, DOC_0007, DOC_0047,...",5.139049,3703,144,
8,20240531542_llm_2,20240531542,llm_2,272,"[20240420207, DOC_0007, 20240419826, 202406053...",5.638485,1713,166,
9,20240723270_fact_budget,20240723270,fact_budget,31,"[20240812818, 20240815487, DOC_0048, 202407232...",0.980074,2370,24,


에러 개수: 0


In [23]:
# Cell 19. RAGEvaluator 생성

In [24]:
evaluator = RAGEvaluator(
    auto_download_nltk=False,
    use_nltk_tokenizer=False,
)

print("RAGEvaluator ready")

RAGEvaluator ready


# Cell 20. 전체 평가 실행

In [25]:
with log_step("Evaluate all metrics"):
    metrics = evaluator.evaluate_all(
        rag_outputs,
        k=BASELINE_CONFIG["top_k"],
    )

metrics

[Evaluate all metrics] start
[Evaluate all metrics] done | elapsed=28s


{'hits@5': np.float64(0.55),
 'precision@5': np.float64(0.11000000000000001),
 'recall@5': np.float64(0.55),
 'mrr@5': np.float64(0.475),
 'retrieval_eval_count': 20,
 'retrieval_eval_skipped_count': 0,
 'avg_bleu': 0.009225972380367882,
 'avg_rougeL': 0.18595238095238095,
 'avg_token_f1': 0.07771066966892115,
 'avg_keyword_group_recall': 0.25916666666666666,
 'exact_keyword_group_match_rate': 0.05,
 'avg_matched_keyword_groups': 0.95,
 'avg_total_keyword_groups': 2.85,
 'avg_retrieval_latency_sec': 0.027366297756088898,
 'avg_generation_latency_sec': 4.8321209686400834,
 'avg_total_latency_sec': 4.867204390402184,
 'p50_total_latency_sec': 5.1390488520264626,
 'p95_total_latency_sec': 9.376959030982107,
 'avg_input_tokens': 2385.55,
 'avg_output_tokens': 140.6,
 'avg_total_tokens': 2526.15,
 'total_tokens': 50523.0,
 'avg_cost_per_query': 0.0,
 'total_cost': 0.0}

# Cell 21. 질문 유형별 평가

In [26]:
with log_step("Evaluate by question_type"):
    metrics_by_question_type = evaluator.evaluate_by_group(
        rag_outputs,
        group_key="question_type",
        k=BASELINE_CONFIG["top_k"],
    )

metrics_by_question_type

[Evaluate by question_type] start
[Evaluate by question_type] done | elapsed=10s


{'llm_1': {'hits@5': np.float64(0.5),
  'precision@5': np.float64(0.1),
  'recall@5': np.float64(0.5),
  'mrr@5': np.float64(0.5),
  'retrieval_eval_count': 8,
  'retrieval_eval_skipped_count': 0,
  'avg_bleu': 0.017784708874941876,
  'avg_rougeL': 0.125,
  'avg_token_f1': 0.10391049792384062,
  'avg_keyword_group_recall': 0.3770833333333333,
  'exact_keyword_group_match_rate': 0.125,
  'avg_matched_keyword_groups': 1.25,
  'avg_total_keyword_groups': 3.375,
  'avg_retrieval_latency_sec': 0.02715682314010337,
  'avg_generation_latency_sec': 6.093779354385333,
  'avg_total_latency_sec': 6.128580633492675,
  'p50_total_latency_sec': 7.842709079035558,
  'p95_total_latency_sec': 15.377135135931894,
  'avg_input_tokens': 2372.5,
  'avg_output_tokens': 179.125,
  'avg_total_tokens': 2551.625,
  'total_tokens': 20413.0,
  'avg_cost_per_query': 0.0,
  'total_cost': 0.0,
  'num_rows': 8},
 'fact_budget': {'hits@5': np.float64(1.0),
  'precision@5': np.float64(0.19999999999999998),
  'recall@5'

In [27]:
evaluator.save_metrics(
    metrics_by_question_type,
    str(METRICS_BY_QTYPE_PATH),
)

평가 결과 저장 완료: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_by_question_type.json


# Cell 22. source_type / answer_format / file_type별 평가

In [28]:
with log_step("Evaluate by source_type"):
    metrics_by_source_type = evaluator.evaluate_by_group(
        rag_outputs,
        group_key="source_type",
        k=BASELINE_CONFIG["top_k"],
    )

with log_step("Evaluate by answer_format"):
    metrics_by_answer_format = evaluator.evaluate_by_group(
        rag_outputs,
        group_key="answer_format",
        k=BASELINE_CONFIG["top_k"],
    )

with log_step("Evaluate by file_type"):
    metrics_by_file_type = evaluator.evaluate_by_group(
        rag_outputs,
        group_key="file_type",
        k=BASELINE_CONFIG["top_k"],
    )

metrics_by_source_type, metrics_by_answer_format, metrics_by_file_type

[Evaluate by source_type] start
[Evaluate by source_type] done | elapsed=0s
[Evaluate by answer_format] start
[Evaluate by answer_format] done | elapsed=0s
[Evaluate by file_type] start
[Evaluate by file_type] done | elapsed=0s


({'document': {'hits@5': np.float64(0.35714285714285715),
   'precision@5': np.float64(0.07142857142857142),
   'recall@5': np.float64(0.35714285714285715),
   'mrr@5': np.float64(0.35714285714285715),
   'retrieval_eval_count': 14,
   'retrieval_eval_skipped_count': 0,
   'avg_bleu': 0.013179960543382689,
   'avg_rougeL': 0.10714285714285714,
   'avg_token_f1': 0.11101524238417307,
   'avg_keyword_group_recall': 0.37023809523809526,
   'exact_keyword_group_match_rate': 0.07142857142857142,
   'avg_matched_keyword_groups': 1.3571428571428572,
   'avg_total_keyword_groups': 3.642857142857143,
   'avg_retrieval_latency_sec': 0.027124538726639003,
   'avg_generation_latency_sec': 6.376399158142574,
   'avg_total_latency_sec': 6.41062006192182,
   'p50_total_latency_sec': 5.641499083954841,
   'p95_total_latency_sec': 9.376959030982107,
   'avg_input_tokens': 2174.785714285714,
   'avg_output_tokens': 187.35714285714286,
   'avg_total_tokens': 2362.1428571428573,
   'total_tokens': 33070.0

In [29]:
evaluator.save_metrics(
    metrics_by_source_type,
    str(METRICS_BY_SOURCE_TYPE_PATH),
)

evaluator.save_metrics(
    metrics_by_answer_format,
    str(METRICS_BY_ANSWER_FORMAT_PATH),
)

evaluator.save_metrics(
    metrics_by_file_type,
    str(METRICS_BY_FILE_TYPE_PATH),
)

평가 결과 저장 완료: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_by_source_type.json
평가 결과 저장 완료: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_by_answer_format.json
평가 결과 저장 완료: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_by_file_type.json


# Cell 23. 검색 실패 케이스 분석

In [30]:
with log_step("Extract retrieval failure cases"):
    retrieval_failures = evaluator.get_retrieval_failure_cases(
        rag_outputs,
        k=BASELINE_CONFIG["top_k"],
    )

print("검색 실패 케이스 수:", len(retrieval_failures))

evaluator.save_rows_as_csv(
    retrieval_failures,
    str(RETRIEVAL_FAILURE_PATH),
)

pd.DataFrame(retrieval_failures).head()

[Extract retrieval failure cases] start
[Extract retrieval failure cases] done | elapsed=0s
검색 실패 케이스 수: 9
CSV 저장 완료: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_retrieval_failures.csv / 9건


,qid,doc_id,question_type,source_type,answer_format,project_name,organization,question,reference,retrieved_ids,response
0,20240430896_llm_2,20240430896,llm_2,document,list,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),경상북도 봉화군,"자가진단표와 관련된 사업범위는 무엇이며, 제출해야 할 별지서식 번호는 어느 것인가요?","사업범위는 자가진단표 분야에 대한 사업수행실적 평가표이며, 관련 별지서식은 별지서식...","[20240531160, DOC_0026, 20240903676, R25BK0056...",자가진단표와 관련된 사업범위는 다음과 같습니다:\n\n1. 핵심투입인력 필수자격요건...
1,20240430918_llm_2,20240430918,llm_2,document,period,도시계획위원회 통합관리시스템 구축용역,인천광역시,"과업내용 변경이나 계약금액·기간 조정이 필요할 때 어떤 절차로 요청해야 하고, 그 ...",과업내용 변경 및 계약금액·계약기간 조정이 필요하면 계약상대자는 국가기관등의 장에게...,"[R25BK00604826, 20240414353, 20240419826, 2024...","과업내용 변경이나 계약금액·기간 조정이 필요할 때, 다음과 같은 절차를 거쳐야 합니..."
2,20240523741_llm_1,20240523741,llm_1,document,summary,축산물이력관리시스템 개선(정보화 사업),축산물품질평가원,이 사업의 주관기관은 어디인가요?,축산물품질평가원,"[20240419826, 20240611568, 20241218257, 202410...",주관기관은 한국건강가정진흥원입니다.
3,20240531542_llm_2,20240531542,llm_2,document,list,(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축」 위탁용역,국립중앙의료원,"이 사업이 추진되는 배경(주요 문제)은 무엇이고, 사업범위에는 어떤 내용이 포함되나요?",추진배경은 응급실 섭외 지연(병원 전 단계에서 치료 가능한 응급실 섭외 문제)과 병...,"[20240420207, DOC_0007, 20240419826, 202406053...",이 사업이 추진되는 배경(주요 문제)은 미술진흥법에 따른 통합미술정보시스템 구축 및...
4,20240903688_llm_1,20240903688,llm_1,document,list,JST 공유대학(원) xAPI기반 LRS시스템 구축,전북대학교,이 사업에서 구축하려는 시스템의 기술 기반은 무엇인가요?,JST 공유대학(원) 내에 xAPI 기반의 LRS(학습기록저장소) 시스템을 구축하는...,"[20240541684, 20240539319, 20240430918, DOC_00...",이 사업에서 구축하려는 시스템의 기술 기반은 GIS(Geographic Inform...


# Cell 24. 키워드 실패 케이스 분석

In [31]:
with log_step("Extract keyword failure cases"):
    keyword_failures = evaluator.get_keyword_failure_cases(
        rag_outputs,
        threshold=1.0,
    )

print("키워드 실패 케이스 수:", len(keyword_failures))

evaluator.save_rows_as_csv(
    keyword_failures,
    str(KEYWORD_FAILURE_PATH),
)

pd.DataFrame(keyword_failures).head()

[Extract keyword failure cases] start
[Extract keyword failure cases] done | elapsed=0s
키워드 실패 케이스 수: 19
CSV 저장 완료: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_keyword_failures.csv / 19건


,qid,doc_id,question_type,source_type,answer_format,project_name,organization,question,reference,response,keyword_group_recall,matched_keyword_group_count,total_keyword_group_count,matched_groups,missed_groups
0,20240330003_llm_1,20240330003,llm_1,document,list,2024년 벤처확인종합관리시스템 기능 고도화 용역사업 입찰공고,(사)벤처기업협회,2024년 벤처확인종합관리시스템 기능 고도화 사업에서 고도화하거나 신규로 구축하는 ...,"복수의결권주식의 발행 보고 업무처리 시스템 구축, 벤처기업법에 따라 부여된 스톡옵션...",2024년 벤처확인종합관리시스템 기능 고도화 사업에서 고도화하거나 신규로 구축하는 ...,0.666667,2,3,"[[복수의결권주식, 복수 의결권], [스톡옵션, 주식매수선택권, 스톡 옵션]]","[[성과조건부주식교부, 성과조건부주식교부계약, RS]]"
1,20240404154_fact_budget,20240404154,fact_budget,metadata,money,2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역,서울특별시,서울특별시에서 발주한 '2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 ...,"493,763,000원 입니다.","배정 예산은 금493,763천원(부가가치세 포함)입니다.",0.000000,0,1,[],"[[493,763,000원, 493763000원, 4억9천3백7십6만원]]"
2,20240408682_llm_1,20240408682,llm_1,document,period,2024년도 GKL 그룹웨어 시스템 구축 용역,그랜드코리아레저(주),GKL 그룹웨어 시스템 구축사업의 사업기간은 언제까지인가요?,계약일로부터 6개월 이내입니다.,GKL 그룹웨어 시스템 구축사업의 사업기간은 계약체결일로부터 6개월 (안정화 1개월...,0.500000,1,2,"[[6개월, 6개월 이내, 6개월 이내입니다]]","[[계약일로부터, 계약일]]"
3,20240430896_llm_2,20240430896,llm_2,document,list,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),경상북도 봉화군,"자가진단표와 관련된 사업범위는 무엇이며, 제출해야 할 별지서식 번호는 어느 것인가요?","사업범위는 자가진단표 분야에 대한 사업수행실적 평가표이며, 관련 별지서식은 별지서식...",자가진단표와 관련된 사업범위는 다음과 같습니다:\n\n1. 핵심투입인력 필수자격요건...,0.666667,2,3,"[[자가진단표, 자가 진단표, 정량 평가], [사업범위, 사업수행실적 평가표]]","[[별지서식 11호, 별지서식11호, 11호]]"
4,20240430918_fact_budget,20240430918,fact_budget,metadata,money,도시계획위원회 통합관리시스템 구축용역,인천광역시,인천광역시에서 발주한 '도시계획위원회 통합관리시스템 구축용역' 사업의 배정 예산은 ...,"150,000,000원 입니다.","죄송합니다, 현재 문서에서는 특정 사업의 예산 정보나 배정 예산에 대한 정보가 제공...",0.000000,0,1,[],"[[150,000,000원, 150000000원, 1억5천만원]]"


# Cell 25. row별 keyword score 부착 후 저장

In [32]:
with log_step("Attach keyword scores"):
    scored_outputs = evaluator.attach_keyword_scores(rag_outputs)

evaluator.save_rows_as_json(
    scored_outputs,
    str(RAG_OUTPUT_SCORED_PATH),
)

print("scored output 저장:", RAG_OUTPUT_SCORED_PATH)

[Attach keyword scores] start
[Attach keyword scores] done | elapsed=0s
JSON 저장 완료: /home/user1/RFP-RAG-Extractor/data/processed/eval/rag_outputs_baseline_section_sample_20_scored.json / 20건
scored output 저장: /home/user1/RFP-RAG-Extractor/data/processed/eval/rag_outputs_baseline_section_sample_20_scored.json


# Cell 26. 요약 CSV 생성

In [33]:
summary_rows = []

for row in scored_outputs:
    retrieved_ids_topk = row.get("retrieved_ids", [])[:BASELINE_CONFIG["top_k"]]

    summary_rows.append({
        "qid": row.get("qid"),
        "doc_id": row.get("doc_id"),
        "question_type": row.get("question_type"),
        "source_type": row.get("source_type"),
        "answer_format": row.get("answer_format"),
        "file_type": row.get("file_type"),
        "project_name": row.get("project_name"),
        "organization": row.get("organization"),
        "question": row.get("question"),
        "reference": row.get("reference"),
        "response": row.get("response"),
        "retrieved_ids": json.dumps(row.get("retrieved_ids", []), ensure_ascii=False),
        "retrieval_hit": row.get("doc_id") in set(retrieved_ids_topk),
        "keyword_group_recall": row.get("keyword_group_recall"),
        "matched_keyword_group_count": row.get("matched_keyword_group_count"),
        "total_keyword_group_count": row.get("total_keyword_group_count"),
        "missed_groups": json.dumps(row.get("missed_groups", []), ensure_ascii=False),
        "retrieval_latency_sec": row.get("retrieval_latency_sec"),
        "generation_latency_sec": row.get("generation_latency_sec"),
        "total_latency_sec": row.get("total_latency_sec"),
        "input_tokens": row.get("input_tokens"),
        "output_tokens": row.get("output_tokens"),
        "error": row.get("error", ""),
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    SUMMARY_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("요약 CSV 저장:", SUMMARY_CSV_PATH)
display(summary_df.head())

요약 CSV 저장: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_summary.csv


,qid,doc_id,question_type,source_type,answer_format,file_type,project_name,organization,question,reference,...,keyword_group_recall,matched_keyword_group_count,total_keyword_group_count,missed_groups,retrieval_latency_sec,generation_latency_sec,total_latency_sec,input_tokens,output_tokens,error
0,20240330003_llm_1,20240330003,llm_1,document,list,hwp,2024년 벤처확인종합관리시스템 기능 고도화 용역사업 입찰공고,(사)벤처기업협회,2024년 벤처확인종합관리시스템 기능 고도화 사업에서 고도화하거나 신규로 구축하는 ...,"복수의결권주식의 발행 보고 업무처리 시스템 구축, 벤처기업법에 따라 부여된 스톡옵션...",...,0.666667,2,3,"[[""성과조건부주식교부"", ""성과조건부주식교부계약"", ""RS""]]",0.028735,15.341774,15.377135,2235,465,
1,20240404154_fact_budget,20240404154,fact_budget,metadata,money,pdf,2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역,서울특별시,서울특별시에서 발주한 '2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 ...,"493,763,000원 입니다.",...,0.000000,0,1,"[[""493,763,000원"", ""493763000원"", ""4억9천3백7십6만원""]]",0.030511,1.012225,1.051589,2701,26,
2,20240408682_llm_1,20240408682,llm_1,document,period,hwp,2024년도 GKL 그룹웨어 시스템 구축 용역,그랜드코리아레저(주),GKL 그룹웨어 시스템 구축사업의 사업기간은 언제까지인가요?,계약일로부터 6개월 이내입니다.,...,0.500000,1,2,"[[""계약일로부터"", ""계약일""]]",0.026596,1.410877,1.444478,2221,39,
3,20240430896_llm_2,20240430896,llm_2,document,list,hwp,봉화군 재난통합관리시스템 고도화 사업(협상)(긴급),경상북도 봉화군,"자가진단표와 관련된 사업범위는 무엇이며, 제출해야 할 별지서식 번호는 어느 것인가요?","사업범위는 자가진단표 분야에 대한 사업수행실적 평가표이며, 관련 별지서식은 별지서식...",...,0.666667,2,3,"[[""별지서식 11호"", ""별지서식11호"", ""11호""]]",0.026746,3.922587,3.956837,2462,113,
4,20240430918_fact_budget,20240430918,fact_budget,metadata,money,hwp,도시계획위원회 통합관리시스템 구축용역,인천광역시,인천광역시에서 발주한 '도시계획위원회 통합관리시스템 구축용역' 사업의 배정 예산은 ...,"150,000,000원 입니다.",...,0.000000,0,1,"[[""150,000,000원"", ""150000000원"", ""1억5천만원""]]",0.027289,1.487514,1.525542,3451,38,


# Cell 27. 주요 지표 출력

In [34]:
top_k = BASELINE_CONFIG["top_k"]

print("===== Baseline RAG Evaluation =====")
print(f"LLM: {BASELINE_CONFIG['llm_model_name']}")
print(f"Embedding: {BASELINE_CONFIG['embedding_model_name']}")
print(f"Vector DB: {BASELINE_CONFIG['vector_db']}")
print(f"Chunking: {BASELINE_CONFIG['chunking_strategy']}")
print(f"Top-K: {top_k}")
print(f"Sample Size: {BASELINE_CONFIG['sample_size']}")

print("\n--- Retrieval ---")
for key in [
    f"hits@{top_k}",
    f"precision@{top_k}",
    f"recall@{top_k}",
    f"mrr@{top_k}",
    "retrieval_eval_count",
    "retrieval_eval_skipped_count",
]:
    print(f"{key}: {metrics.get(key)}")

print("\n--- Generation ---")
for key in [
    "avg_bleu",
    "avg_rougeL",
    "avg_token_f1",
]:
    print(f"{key}: {metrics.get(key)}")

print("\n--- Keyword ---")
for key in [
    "avg_keyword_group_recall",
    "exact_keyword_group_match_rate",
    "avg_matched_keyword_groups",
    "avg_total_keyword_groups",
]:
    print(f"{key}: {metrics.get(key)}")

print("\n--- Efficiency ---")
for key in [
    "avg_retrieval_latency_sec",
    "avg_generation_latency_sec",
    "avg_total_latency_sec",
    "p50_total_latency_sec",
    "p95_total_latency_sec",
    "avg_input_tokens",
    "avg_output_tokens",
    "avg_total_tokens",
    "total_tokens",
    "avg_cost_per_query",
    "total_cost",
]:
    print(f"{key}: {metrics.get(key)}")

print("\n--- Failures ---")
print("retrieval_failures:", len(retrieval_failures))
print("keyword_failures:", len(keyword_failures))

===== Baseline RAG Evaluation =====
LLM: Qwen/Qwen2.5-1.5B-Instruct
Embedding: nlpai-lab/KURE-v1
Vector DB: FAISS
Chunking: section
Top-K: 5
Sample Size: 20

--- Retrieval ---
hits@5: 0.55
precision@5: 0.11000000000000001
recall@5: 0.55
mrr@5: 0.475
retrieval_eval_count: 20
retrieval_eval_skipped_count: 0

--- Generation ---
avg_bleu: 0.009225972380367882
avg_rougeL: 0.18595238095238095
avg_token_f1: 0.07771066966892115

--- Keyword ---
avg_keyword_group_recall: 0.25916666666666666
exact_keyword_group_match_rate: 0.05
avg_matched_keyword_groups: 0.95
avg_total_keyword_groups: 2.85

--- Efficiency ---
avg_retrieval_latency_sec: 0.027366297756088898
avg_generation_latency_sec: 4.8321209686400834
avg_total_latency_sec: 4.867204390402184
p50_total_latency_sec: 5.1390488520264626
p95_total_latency_sec: 9.376959030982107
avg_input_tokens: 2385.55
avg_output_tokens: 140.6
avg_total_tokens: 2526.15
total_tokens: 50523.0
avg_cost_per_query: 0.0
total_cost: 0.0

--- Failures ---
retrieval_failur

# Cell 28. 다음 실험을 위한 메모 저장

In [35]:
experiment_summary = {
    "config": BASELINE_CONFIG,
    "paths": {
        "section_chunk_path": str(SECTION_CHUNK_PATH),
        "eval_dataset_path": str(EVAL_DATASET_PATH),
        "eval_sample_path": str(EVAL_SAMPLE_PATH),
        "vector_dir": str(BASELINE_VECTOR_DIR),
        "rag_output_path": str(RAG_OUTPUT_PATH),
        "metrics_path": str(METRICS_PATH),
        "summary_csv_path": str(SUMMARY_CSV_PATH),
    },
    "metrics": metrics,
    "num_rag_outputs": len(rag_outputs),
    "num_retrieval_failures": len(retrieval_failures),
    "num_keyword_failures": len(keyword_failures),
}

EXPERIMENT_SUMMARY_PATH = REPORT_DIR / "baseline_section_sample20_experiment_summary.json"

evaluator.save_metrics(
    experiment_summary,
    str(EXPERIMENT_SUMMARY_PATH),
)

print("실험 요약 저장:", EXPERIMENT_SUMMARY_PATH)

평가 결과 저장 완료: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_experiment_summary.json
실험 요약 저장: /home/user1/RFP-RAG-Extractor/reports/evaluation/baseline_section_sample20_experiment_summary.json
